In [4]:
import ee
import geemap
import pandas as pd
import numpy as np


# Initialize the library
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

# Initialize the map
Map = geemap.Map()

# elevation

In [2]:
dataset_etopo = ee.Image('NOAA/NGDC/ETOPO1')
elevation = dataset_etopo.select('bedrock')

elevation_vis = {
    'min': -7000.0,
    'max': 3000.0,
    'palette': ['011de2', 'afafaf', '3603ff', 'fff477', 'b42109'],
}

# Set center for Elevation view
Map.setCenter(-37.62, 25.8, 2)
Map.addLayer(elevation, elevation_vis, 'Elevation')

# landforms

In [3]:
dataset_landforms = ee.Image('CSP/ERGo/1_0/Global/ALOS_landforms')
landforms = dataset_landforms.select('constant')

landforms_vis = {
    'min': 11.0,
    'max': 42.0,
    'palette': [
        '141414', '383838', '808080', 'ebeb8f', 'f7d311', 'aa0000', 'd89382',
        'ddc9c9', 'dccdce', '1c6330', '68aa63', 'b5c98e', 'e1f0e5', 'a975ba',
        '6f198c'
    ],
}

# Set center for Landforms view (this will override the previous center)
Map.setCenter(-105.58, 40.5498, 11)
Map.addLayer(landforms, landforms_vis, 'Landforms')

In [5]:
# 1. Load the data
cal_data = pd.read_csv("../california_coord.csv")

cal_coords = cal_data[["longitude","latitude"]].values

print(f"coords shape:    ,{cal_coords.shape}")

coords shape:    ,(10000, 2)


In [6]:
# 1. Load the data
ark_data = pd.read_csv("../arkansas_coord.csv")

ark_coords = ark_data[["longitude","latitude"]].values

print(f"coords shape:    ,{ark_coords.shape}")

coords shape:    ,(10000, 2)


## california 
change cal_coords and drive file names for arkansas

In [28]:
# Convert ark_coords (N, 2) to a list of ee.Feature
pts = [ee.Feature(ee.Geometry.Point(coord[0], coord[1])) for coord in cal_coords]
points_fc = ee.FeatureCollection(pts)

#### soil and topography

In [29]:
# Soil: Organic Carbon, Texture, and pH (Top layer selected)
carbon = ee.Image("OpenLandMap/SOL/SOL_ORGANIC-CARBON_USDA-6A1C_M/v02").select(0).rename('carbon')
texture = ee.Image("OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02").select(0).rename('texture')
ph = ee.Image("OpenLandMap/SOL/SOL_PH-H2O_USDA-4C1A2A_M/v02").select(0).rename('ph')

# Topography: Elevation and Landforms
elevation = ee.Image("NOAA/NGDC/ETOPO1").select('bedrock').rename('elevation')
landforms = ee.Image("CSP/ERGo/1_0/Global/ALOS_landforms").select('constant').rename('landforms')

static_stack = ee.Image.cat([carbon, texture, ph, elevation, landforms])

# 1. Apply resampling to the images before stacking
# This creates smoother gradients when sampling at high resolution
static_stack_smooth = static_stack.resample('bilinear')
# 2. Sample at 10m to match your Sentinel-2 resolution
sampled_fc = static_stack_smooth.sampleRegions(
    collection=points_fc,
    scale=10, # Matches Sentinel-2 high-res bands
    geometries=True,
    tileScale=4 # Use tileScale to handle the computational load of 10m sampling
)

# 3. Export
task = ee.batch.Export.table.toDrive(
    collection=sampled_fc,
    description='California_Soil_Topography',
    fileFormat='CSV',
    folder='EarthEngine_Exports'
)
task.start()

#### climate

**climate feature choice reason:
**They capture the three dominant drivers of crop behavior:

Energy (temperature)
Water supply (precipitation)
Atmospheric demand / stress (dewpoint)



 
🌾 Crop classification + climate features
You et al., 2017 – “Deep Gaussian Process for Crop Mapping”
Temperature + precipitation are dominant predictors
Jiang et al., 2020 – “Crop classification using satellite + weather data”
Best performance with temperature + precipitation + humidity-related variables

🌿 Agronomy / plant physiology
Allen et al., 1998 – FAO Irrigation & Drainage Paper 56
Defines evapotranspiration using:
Temperature
Humidity (dewpoint proxy)
Wind
Radiation
👉 Confirms dewpoint + temp combo is critical
Lobell & Burke, 2010 – Climate impacts on agriculture
Temperature + water availability dominate yield variability

🛰️ Remote sensing + ML
Rustowicz et al., 2019 – Crop type classification with deep learning
Adding weather improves performance
Key variables: temperature + precipitation
Khaki et al., 2020 – Deep learning for yield prediction
Uses temperature + precipitation + humidity proxies

In [30]:
# 2. Climate Configuration
start_date = ee.Date('2021-01-01')
variables = ['mean_2m_air_temperature', 'total_precipitation', 'dewpoint_2m_temperature']
climate_stack = ee.Image()

# 3. Create the 36-step Stack
for i in range(36):
    d1 = start_date.advance(i * 10, 'day')
    d2 = d1.advance(10, 'day')
    
    # Filter for the 10-day window and take the median
    interval_median = ee.ImageCollection("ECMWF/ERA5/DAILY") \
        .filterDate(d1, d2) \
        .select(variables) \
        .median()
    
    # Rename bands to stay organized: temp_0, precip_0, dew_0, etc.
    interval_median = interval_median.rename([
        f'temp_{i}', 
        f'precip_{i}', 
        f'dew_{i}'
    ])
    
    # Add to the master stack
    climate_stack = climate_stack.addBands(interval_median)

# Remove the initial empty 'constant' band
climate_stack = climate_stack.select('temp_.*|precip_.*|dew_.*')




start_date = ee.Date('2021-01-01')

# The correct band names for ERA5_LAND/DAILY_AGGR
target_bands = ['temperature_2m', 'total_precipitation_sum', 'dewpoint_temperature_2m']

# Initialize with step 0
first_interval = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
    .filterDate('2021-01-01', '2021-01-11') \
    .select(target_bands) \
    .median() \
    .rename(['temp_0', 'precip_0', 'dew_0'])

climate_stack = first_interval

# 2. Loop through the remaining 35 intervals
for i in range(1, 36):
    d1 = start_date.advance(i * 10, 'day')
    d2 = d1.advance(10, 'day')
    
    interval_median = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
        .filterDate(d1, d2) \
        .select(target_bands) \
        .median() \
        .rename([f'temp_{i}', f'precip_{i}', f'dew_{i}'])
    
    climate_stack = climate_stack.addBands(interval_median)

# 3. Verify and Sample
band_names = climate_stack.bandNames().getInfo()
print(f"Verified bands to export: {len(band_names)}") 

sampled_climate = climate_stack.sampleRegions(
    collection=points_fc,
    scale=11132, 
    geometries=True
)

# 4. Export
task = ee.batch.Export.table.toDrive(
    collection=sampled_climate,
    description='California_Climate',
    fileFormat='CSV',
    folder='EarthEngine_Exports',
    selectors=['.geo'] + band_names
)

task.start()
print("Export started. Band names are now synchronized with ERA5 Land Aggr.")


Verified bands to export: 108
Export started. Band names are now synchronized with ERA5 Land Aggr.


# read imported data

## merge raw data with new features

In [5]:
cal_raw = pd.read_csv('../california_coord.csv')
cal_climate = pd.read_csv('../California_Climate.csv')
cal_static = pd.read_csv('../California_Soil_Topography.csv')

ark_raw = pd.read_csv('../arkansas_coord.csv')
ark_climate = pd.read_csv('../Arkansas_Climate.csv')
ark_static = pd.read_csv('../Arkansas_Soil_Topography.csv')



In [8]:
cal_climate.columns

Index(['.geo', 'temp_0', 'precip_0', 'dew_0', 'temp_1', 'precip_1', 'dew_1',
       'temp_2', 'precip_2', 'dew_2',
       ...
       'dew_32', 'temp_33', 'precip_33', 'dew_33', 'temp_34', 'precip_34',
       'dew_34', 'temp_35', 'precip_35', 'dew_35'],
      dtype='object', length=109)

In [14]:
import pandas as pd
import json
import numpy as np
from scipy.spatial import cKDTree

def spatial_merge(target_df, source_df, prefix=""):
    # 1. Ensure source has clean coordinates
    if '.geo' in source_df.columns:
        source_df['longitude'] = source_df['.geo'].apply(lambda x: json.loads(x)['coordinates'][0])
        source_df['latitude'] = source_df['.geo'].apply(lambda x: json.loads(x)['coordinates'][1])
        source_df = source_df.drop(columns=['.geo'])

    # 2. Build the Tree for the source data (Climate or Static)
    # We use the coordinates as the "search index"
    source_coords = source_df[['longitude', 'latitude']].values
    tree = cKDTree(source_coords)

    # 3. Query the tree using the target (Raw) coordinates
    target_coords = target_df[['longitude', 'latitude']].values
    distances, indices = tree.query(target_coords, k=1)

    # 4. Extract the matching rows and drop their coordinate columns to avoid duplicates
    matched_data = source_df.iloc[indices].reset_index(drop=True)
    matched_data = matched_data.drop(columns=['longitude', 'latitude'])

    # 5. Concatenate horizontally
    return pd.concat([target_df.reset_index(drop=True), matched_data], axis=1)

# --- EXECUTION ---

# Merge Static data into Raw
ark_merged = spatial_merge(ark_raw, ark_static)

# Merge Climate data into the result
ark_final = spatial_merge(ark_merged, ark_climate)

# Merge Static data into Raw
cal_merged = spatial_merge(cal_raw, cal_static)

# Merge Climate data into the result
cal_final = spatial_merge(cal_merged, cal_climate)



In [ ]:
cal_final.to_csv('california_integrated.csv', index=False)
ark_final.to_csv('arkansas_integrated.csv', index=False)

: 